# Team 04 — Site Grid & Side Alignment (Phase 3)

Real buildings are **not** dropped at arbitrary rotations inside a plot. They sit on a site grid,
**parallel to a preferred boundary** (the street frontage / the longest edge / the main-road side).
This notebook replaces the free 5 m sweep + 36 free rotations with **grid-node positions × aligned
orientations**, on a **complex non-orthogonal site**.

What this adds over free placement:

1. A **site grid** derived from a *chosen side* — buildings can only take the {parallel,
   perpendicular} orientations of that side (`site_grid.derive_site_grid` / `aligned_orientations`).
2. **It no longer looks random** — free vs. grid-aligned, side by side, for identical fitness.
3. **Obtuse footprints**: an L tucked into a splayed (non-orthogonal) corner lets its free wing
   follow the *adjacent* side, so its arms spread to the corner's interior angle (> 90°).
4. **Use-driven placement**: a **commercial** building hugs the chosen frontage
   (`boundary_proximity`), while residential leans on view + sun.
5. Two or more buildings placed **together**, each aligned and clearing the rest.

Deterministic (no LLM); only matplotlib is needed.

In [ ]:
from __future__ import annotations
import sys, math
from pathlib import Path

workspace_root = Path.cwd().resolve()
candidate_roots = (workspace_root, workspace_root.parent,
                   workspace_root / 'team_04', workspace_root.parent / 'team_04')
TEAM_ROOT = next((p for p in candidate_roots if (p / 'agent').exists()), None)
if TEAM_ROOT is None:
    raise FileNotFoundError('Run from workspace root, team_04, or team_04/test_notebooks.')
if str(TEAM_ROOT) not in sys.path:
    sys.path.insert(0, str(TEAM_ROOT))

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from agent.tools.site_model import build_site_model
from agent.tools.building_shape_graph import build_shape_model
from agent.tools.parametric_shape import apply_shape_variables, shape_variable_spec
from agent.tools.site_grid import (
    derive_site_grid, aligned_orientations, align_building_to_grid, alignment_score,
    snap_to_grid, corner_interior_angle, corner_wing_rotation,
)
from agent.tools.view_optimizer import (
    sample_valid_placements, optimize_aligned_placement, place_buildings_aligned,
)
print('Team root:', TEAM_ROOT)

## 1. Complex site + the derived grid

A splayed, non-orthogonal pentagon (no right angles). The grid is derived from the **longest side**
by default (the sensible fallback before roads land in Phase 2) — origin + two axes, clipped to the
buildable zone, drawn as grid lines with seed nodes.

In [ ]:
SITE = [[0, 0, 0], [130, 18, 0], [150, 92, 0], [62, 128, 0], [-14, 74, 0], [0, 0, 0]]
site_model = build_site_model(SITE, {'default_setback': 6.0})
grid = derive_site_grid(site_model, spacing=12.0)   # default: longest side

print('chosen side   :', grid['alignment_side_index'], grid['alignment_side_label'])
print('grid angle    :', grid['angle_deg'], 'deg')
print('orientations  :', aligned_orientations(grid), '(parallel, perpendicular)')
print('grid nodes    :', grid['node_count'])

def draw_site(ax, model=site_model, site=SITE):
    ax.add_patch(plt.Polygon([(p[0], p[1]) for p in site[:-1]], fc='#f0ede6', ec='#555', lw=2, zorder=1))
    bz = (model.get('setbacks') or {}).get('buildable_boundary')
    if bz:
        ax.add_patch(plt.Polygon([(p[0], p[1]) for p in bz[:-1]], fc='none', ec='#27ae60', lw=1.3, ls='--', zorder=2))
    ax.set_aspect('equal')

def draw_grid(ax, g, nodes=True):
    for ln in g['grid_lines']:
        ax.plot([ln[0][0], ln[1][0]], [ln[0][1], ln[1][1]], color='#b0c4de', lw=0.6, zorder=2)
    if nodes:
        xs = [n[0] for n in g['grid_nodes']]; ys = [n[1] for n in g['grid_nodes']]
        ax.scatter(xs, ys, s=8, color='#5d6d7e', zorder=3)
    # highlight the chosen side
    i = g['alignment_side_index']; coords = [(p[0], p[1]) for p in SITE[:-1]]
    a = coords[i]; b = coords[(i + 1) % len(coords)]
    ax.plot([a[0], b[0]], [a[1], b[1]], color='#e67e22', lw=4, zorder=4, label='chosen side')

fig, ax = plt.subplots(figsize=(8, 7))
draw_site(ax); draw_grid(ax, grid)
ax.legend(loc='upper right'); ax.set_title('Complex site + grid aligned to the chosen (longest) side')
ax.set_xlim(-25, 165); ax.set_ylim(-15, 140)
plt.show()

## 1b. Adaptive grid — the building orients to the local grid (rigid, realistic)

The uniform grid uses **one rigid angle**; on a splayed plot that single angle drifts off the
tapering sides. `derive_adaptive_site_grid` fits a **transfinite (Coons) patch** to the site's four
edge-chains, so the grid's **local axis angle varies across the site** (`angle_range_deg`).

We use that warped grid as an **orientation field**, not as a rubber sheet: the building stays a
**rigid footprint** — straight walls, exact shape, exact area — and `align_building_to_local_grid`
just *rotates* it to the local grid direction at its node. So a building near the road aligns to the
road; one deeper in aligns to the local grid there. The layout adapts to the site while **every
building stays a real building**. (An earlier experiment that deformed the footprint through the
patch was removed — it over-warped complex shapes and blew up their area, which no real building does.)

In [ ]:
from agent.tools.site_grid import (
    derive_adaptive_site_grid, align_building_to_local_grid, local_grid_orientation,
)
from agent.tools.view_analysis import _coerce_polygon_2d

# Adaptive grid = a transfinite (Coons) map whose LOCAL axis angle follows the site
# taper. We use it only as an ORIENTATION FIELD: the building stays a rigid shape
# (straight walls) and is rotated to the local grid direction - never deformed.
agrid = derive_adaptive_site_grid(site_model, spacing=12.0)
print(f"uniform  grid: single angle {grid['angle_deg']:.1f} deg")
print(f"adaptive grid: local angle varies {agrid['angle_range_deg']:.1f} deg across the site")

def draw_warped_grid(ax, g, color='#7fbf3f'):
    for ln in g['grid_lines']:                       # full warped polylines
        ax.plot([p[0] for p in ln], [p[1] for p in ln], color=color, lw=0.8, zorder=2)

def draw_site_red(ax, site=SITE):
    ax.plot([p[0] for p in site], [p[1] for p in site], color='#e23b2e', lw=2.5, zorder=5)
    ax.set_aspect('equal')

def road_seg(g):
    i = g['alignment_side_index']; cs = [(p[0], p[1]) for p in SITE[:-1]]
    return cs[i], cs[(i + 1) % len(cs)]

def draw_road(ax, g, label=True):
    a, b = road_seg(g)
    ax.plot([a[0], b[0]], [a[1], b[1]], color='#f08a24', lw=6, zorder=4)
    if label:
        ax.annotate('main road => chosen side', xy=((a[0]+b[0])/2, (a[1]+b[1])/2),
                    xytext=(a[0] + 4, a[1] - 6),
                    rotation=math.degrees(math.atan2(b[1]-a[1], b[0]-a[0])),
                    fontsize=8, color='#7a4a00', zorder=7)

def draw_building(ax, placed, color='#27408b'):
    ax.add_patch(plt.Polygon([(p[0], p[1]) for p in placed[:-1]], closed=True,
                 fc=color, ec='#10204a', lw=1.4, hatch='||', alpha=0.85, zorder=6))

_site_poly = _coerce_polygon_2d(SITE)

def place_local_inside(base, g, target, extra_rotation_deg=0.0):
    """RIGID placement: rotate the footprint to the local grid direction at the
    nearest node where it fits fully inside the site. The shape is preserved exactly
    (straight walls, same area) - only its orientation/position adapt."""
    for nd in sorted(g['grid_nodes'], key=lambda n: (n[0]-target[0])**2 + (n[1]-target[1])**2):
        placed = align_building_to_local_grid(base, g, nd, extra_rotation_deg=extra_rotation_deg)
        if _site_poly.contains(_coerce_polygon_2d(placed)):
            return placed, nd
    return None, None

coords = [(p[0], p[1]) for p in SITE[:-1]]
cen = [sum(c[0] for c in coords) / len(coords), sum(c[1] for c in coords) / len(coords)]
u_base = [[x, y, 0.0] for x, y in build_shape_model(
    area=700.0, building_type='U', building_depth=14.0, shape_ratio=0.5).polygon.exterior.coords]

placed_local, node_a = place_local_inside(u_base, agrid, cen)
print(f"U placed rigidly, oriented to local grid angle {local_grid_orientation(agrid, node_a):.1f} deg "
      f"(shape preserved - {len(placed_local)} verts, same as base {len(u_base)})")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
# Left: uniform grid, building at the single chosen-side angle (rigid).
draw_site(axes[0]); draw_grid(axes[0], grid, nodes=False)
a, b = road_seg(grid); axes[0].plot([a[0], b[0]], [a[1], b[1]], color='#e67e22', lw=4, zorder=4)
unode = snap_to_grid(cen, grid)
placed_uniform = align_building_to_grid(u_base, grid, unode, aligned_orientations(grid)[0])
draw_building(axes[0], placed_uniform, color='#7f8c8d')
axes[0].set_title(f'Uniform grid - rigid U at one fixed angle ({grid["angle_deg"]:.0f} deg)')
# Right: adaptive grid, the SAME rigid U oriented to the LOCAL grid direction.
draw_site_red(axes[1]); draw_warped_grid(axes[1], agrid); draw_road(axes[1], agrid)
draw_building(axes[1], placed_local)
axes[1].set_title(f'Adaptive grid - rigid U oriented to the LOCAL grid (shape exact, warp {agrid["angle_range_deg"]:.0f} deg)')
for ax in axes:
    ax.set_xlim(-25, 165); ax.set_ylim(-15, 140)
plt.show()

# Flexibility: the SAME rigid U at different grid positions, each oriented to the
# local grid direction. Position + orientation adapt; the shape never deforms.
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, tg in zip(axes, ([40, 35], [100, 60], [70, 100])):
    placed, nd = place_local_inside(u_base, agrid, tg)
    draw_site_red(ax); draw_warped_grid(ax, agrid); draw_road(ax, agrid, label=False)
    if placed:
        draw_building(ax, placed)
        ax.set_title(f'near {tg} - local angle {local_grid_orientation(agrid, nd):.0f} deg')
    else:
        ax.set_title(f'near {tg} - no in-site fit')
    ax.set_xlim(-25, 165); ax.set_ylim(-15, 140)
fig.suptitle('Same rigid U, different grid positions - orientation adapts, the shape stays exact', y=1.02)
plt.show()

## 1c. Test — the building reacts to the chosen side

The chosen side (the main road) is **always** the grid's bottom (`B`) chain. Re-key the grid to each
side of the *same* plot and place the identical **rigid** U near that side — the building re-orients
to follow whichever side is picked, while keeping its exact shape (panel titles show `B = side`).

This is also the fix for the earlier "looks random" bug: when the chosen side's vertices were not the
sharpest corners, the grid's bottom edge used to drift to other corners. `_select_quad_corners` now
anchors the bottom chain on the chosen side, so `B = side` for **every** side.

In [ ]:
# Re-key the grid to EACH side of the same plot and place the identical RIGID U
# near that side, oriented to the local grid. The building re-orients to follow
# whichever side is chosen and keeps its exact shape.
n_sides = len(SITE) - 1
fig, axes = plt.subplots(1, n_sides, figsize=(3.6 * n_sides, 4.0))
for side, ax in zip(range(n_sides), axes):
    g = derive_adaptive_site_grid(site_model, spacing=12.0, alignment_side=side)
    a, b = road_seg(g)
    target = [(a[0] + b[0]) / 2 + (cen[0] - (a[0] + b[0]) / 2) * 0.35,
              (a[1] + b[1]) / 2 + (cen[1] - (a[1] + b[1]) / 2) * 0.35]
    placed, nd = place_local_inside(u_base, g, target)
    draw_site_red(ax); draw_warped_grid(ax, g); draw_road(ax, g, label=False)
    if placed:
        draw_building(ax, placed)
    b0, b1 = g['corner_indices'][0], g['corner_indices'][1]
    ax.set_title(f"side {side}: B={b0}->{b1}", fontsize=9)
    ax.set_xlim(-25, 165); ax.set_ylim(-15, 140); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Same RIGID U, different chosen side (orange) - it re-orients to that side, shape preserved', y=1.04)
plt.show()

## 1d. Every library shape stays a realistic building

Rigid local-grid placement is shape-agnostic. Every footprint in the library — winged `I L T U H`
and template `Y X O` — is placed at an in-site grid node, **rotated** to the local grid direction.
Each keeps its **exact shape and area** (no deformation) while still aligning to the grid — the
panel titles confirm the placed area equals the base area. This is the contrast with the removed
conforming experiment, which rubber-sheeted X/Y into unrecognisable blobs.

In [ ]:
# Every library shape, placed RIGIDLY and oriented to the local grid. Each keeps
# its exact shape + area; placement only rotates/positions it to follow the grid.
ALL_SHAPES = ['I', 'L', 'T', 'U', 'H', 'Y', 'X', 'O']
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for s, ax in zip(ALL_SHAPES, axes.ravel()):
    base = [[x, y, 0.0] for x, y in build_shape_model(
        area=520.0, building_type=s, building_depth=13.0, shape_ratio=0.5).polygon.exterior.coords]
    base_area = _coerce_polygon_2d(base).area
    placed, nd = place_local_inside(base, agrid, cen)
    draw_site_red(ax); draw_warped_grid(ax, agrid); draw_road(ax, agrid, label=False)
    if placed:
        draw_building(ax, placed)
        area_ok = abs(_coerce_polygon_2d(placed).area - base_area) < 1e-3
        ax.set_title(f'{s}  (area preserved={area_ok})', fontsize=11)
    else:
        ax.set_title(f'{s}  (no in-site fit at centre)', fontsize=11)
    ax.set_xlim(-25, 165); ax.set_ylim(-15, 140); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Every library shape stays a realistic building - rigid, oriented to the grid (I L T U H Y X O)', y=1.0)
plt.tight_layout(); plt.show()

## 2. Free placement vs. grid-aligned placement

The same building, the same site. Left: the old free sweep (any of 36 rotations, anywhere). Right:
restricted to grid nodes × {parallel, perpendicular}. The right reads as *intentional* — every
candidate is parallel to the chosen frontage.

In [ ]:
def base_boundary(btype, area, depth=12.0, ratio=0.5):
    poly = build_shape_model(area=area, building_type=btype, building_depth=depth, shape_ratio=ratio).polygon
    return [[round(float(x), 3), round(float(y), 3), 0.0] for x, y in poly.exterior.coords]

bldg = base_boundary('I', 320.0)
SET = {'default_setback': 6.0}

free = sample_valid_placements(bldg, SITE, rotation_step_degrees=10, grid_step=8.0, site_setbacks=SET)
aligned = sample_valid_placements(bldg, SITE, grid=grid, site_setbacks=SET)
print(f'free candidates    : {len(free)}  (mixed rotations)')
print(f'aligned candidates : {len(aligned)}  (all parallel/perpendicular to the chosen side)')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, cands, title, show_grid in ((axes[0], free[::7], 'FREE sweep (rotates anywhere)', False),
                                     (axes[1], aligned[::3], 'GRID-ALIGNED (parallel to chosen side)', True)):
    draw_site(ax)
    if show_grid: draw_grid(ax, grid, nodes=False)
    for c in cands:
        ax.add_patch(plt.Polygon([(p[0], p[1]) for p in c['boundary'][:-1]],
                                 fc='#2980b9', ec='#1b4f72', lw=0.6, alpha=0.25))
    ax.set_title(title); ax.set_xlim(-25, 165); ax.set_ylim(-15, 140)
plt.show()

## 3. Obtuse corner — a rigid L with obtuse arms follows two non-orthogonal sides

A realistic way for a building to respect a splayed corner: build an **L whose free wing is bent** by
`corner_wing_rotation` so its arms span the corner's interior angle (obtuse on this site), then place
the whole **rigid** footprint oriented to the local grid near that corner. One arm runs along the
chosen side, the other along the adjacent side — and the footprint keeps its exact shape (straight
walls). `place_local_inside` snaps it to the nearest grid node where it sits fully inside the site, so
there is no overflow.

In [ ]:
cs = [(p[0], p[1]) for p in SITE[:-1]]
n = len(cs)
# Pick the site's MOST obtuse corner and key a grid to the side starting there.
angles = [corner_interior_angle(site_model, k) for k in range(n)]
c0 = max(range(n), key=lambda k: angles[k])
theta = angles[c0]
cgrid = derive_adaptive_site_grid(site_model, spacing=12.0, alignment_side=c0)

# Build an L whose free wing bends by (theta - 90) so its arms span the obtuse
# corner angle, then place the whole RIGID footprint oriented to the local grid.
AREA_L = 620.0
spec = shape_variable_spec('L', AREA_L)
n_leaf = len(spec['leaf_wing_indices'])
obtuse_L = apply_shape_variables('L', AREA_L, [13.0, 0.5] + [theta - 90.0] * n_leaf, spec['leaf_wing_indices'])
obtuse_Lb = [[x, y, 0.0] for x, y in obtuse_L.exterior.coords]

corner_pt = cs[c0]
target = [corner_pt[0] + (cen[0] - corner_pt[0]) * 0.40, corner_pt[1] + (cen[1] - corner_pt[1]) * 0.40]
placed, nd = place_local_inside(obtuse_Lb, cgrid, target)
inside = placed is not None
chosen_side = (cs[c0], cs[(c0 + 1) % n])
adjacent_side = (cs[(c0 - 1) % n], cs[c0])
print(f"most obtuse corner = vertex {c0}: interior angle {theta:.0f} deg | rigid obtuse-L inside: {inside} | "
      f"verts {len(placed) if placed else 0} (base {len(obtuse_Lb)}) - shape preserved")

fig, ax = plt.subplots(figsize=(9, 8))
draw_site_red(ax); draw_warped_grid(ax, cgrid)
ax.plot([chosen_side[0][0], chosen_side[1][0]], [chosen_side[0][1], chosen_side[1][1]],
        color='#f08a24', lw=6, zorder=4, label='chosen side')
ax.plot([adjacent_side[0][0], adjacent_side[1][0]], [adjacent_side[0][1], adjacent_side[1][1]],
        color='#9b59b6', lw=5, zorder=4, label='adjacent side')
if placed:
    ax.add_patch(plt.Polygon([(p[0], p[1]) for p in placed[:-1]], closed=True,
                 fc='#16a085', ec='k', lw=1.5, hatch='||', alpha=0.8, zorder=6))
ax.legend(loc='upper right')
ax.set_title(f'Rigid obtuse L ({theta:.0f}° arms) oriented to the local grid - fully inside = {inside}')
ax.set_xlim(-25, 165); ax.set_ylim(-15, 140)
plt.show()

## 4. Use-driven placement — commercial hugs the frontage

Same building, same grid, different `use`. **Commercial / office / retail** buildings pick up a
`boundary_proximity` objective and line the chosen frontage; **residential** leans on view + sun and
sits back. The optimizer is exhaustive over the (small) aligned candidate set, so the best option
is exact.

In [ ]:
from agent.tools.view_analysis import _coerce_polygon_2d
site_poly = _coerce_polygon_2d(SITE)
side_i = grid['alignment_side_index']
coords = [(p[0], p[1]) for p in SITE[:-1]]
ref_line = [list(coords[side_i]), list(coords[(side_i + 1) % len(coords)])]   # the chosen side segment

shop = base_boundary('I', 360.0, depth=14.0)
results = {}
for use in ('commercial', 'residential'):
    res = optimize_aligned_placement(
        base_boundary=shop, site_boundary=SITE, grid=grid, use=use,
        reference_line=ref_line, site_setbacks=SET, saved_option_count=6,
    )
    best = res['options'][0]
    dist = ref_line_dist = _coerce_polygon_2d(best['boundary']).distance(
        __import__('shapely').geometry.LineString(ref_line))
    results[use] = (res, best, dist)
    print(f"{use:>11}: objectives={[c['name'] for c in res['objective_configs']]}")
    print(f"{'':>11}  best distance to chosen frontage = {dist:6.2f} m   (combined {best['combined_score']:.3f})")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, use in zip(axes, ('commercial', 'residential')):
    res, best, dist = results[use]
    draw_site(ax); draw_grid(ax, grid, nodes=False)
    ax.plot([ref_line[0][0], ref_line[1][0]], [ref_line[0][1], ref_line[1][1]], color='#e67e22', lw=4, zorder=4)
    for opt in res['options'][1:]:
        ax.add_patch(plt.Polygon([(p[0], p[1]) for p in opt['boundary'][:-1]], fc='none', ec='#aab', lw=0.8, ls='--', zorder=4))
    c = '#c0392b' if use == 'commercial' else '#2980b9'
    ax.add_patch(plt.Polygon([(p[0], p[1]) for p in best['boundary'][:-1]], fc=c, ec='k', lw=1.5, alpha=0.75, zorder=6))
    ax.set_title(f'{use} — best sits {dist:.1f} m from the frontage')
    ax.set_xlim(-25, 165); ax.set_ylim(-15, 140)
plt.show()

## Summary

- `site_grid.derive_site_grid` builds a placement grid from a **chosen site side** on an arbitrary
  (non-orthogonal) site; `aligned_orientations` is the only orientation set a building may take —
  **no free rotation**.
- `site_grid.derive_adaptive_site_grid` builds a **warped** grid (transfinite/Coons patch) whose
  **local axis angle follows the site taper** (`angle_range_deg`); the bottom chain is always the
  chosen side, so re-keying the grid moves it to follow that side. It is used as an **orientation
  field**, not a rubber sheet.
- **Buildings stay rigid.** `align_building_to_local_grid` rotates a footprint to the local grid
  direction at its node, preserving the **exact shape and area** (straight walls) for every library
  shape (I L T U H Y X O). An L can additionally bend a free wing (`corner_wing_rotation`) to span an
  obtuse corner. (An earlier footprint-*conforming* experiment that deformed the polygon through the
  patch was removed — it over-warped complex shapes and blew up their area, which no real building does.)
- `optimize_aligned_placement` exhaustively ranks grid-node × aligned-orientation candidates with a
  **use-driven** objective mix (commercial → `boundary_proximity`, residential → view + sun), and
  `place_buildings_aligned` sequences several buildings, each aligned and clearing the rest.

Backend: `agent/tools/site_grid.py` (uniform + adaptive grid + rigid local-grid placement) +
`grid_alignment`/`boundary_proximity` objectives and `optimize_aligned_placement`/
`place_buildings_aligned`/grid-aware `sample_valid_placements` in `view_optimizer.py`. Regressions:
`benchmarking/test_site_grid.py` (29 tests). Frontend overlay: `frontend/site/GridOverlay.tsx` via
`POST /tools/{site_grid,aligned_placement}`.

In [ ]:
from agent.tools.sun_analysis import compute_sun_vectors
sun_vectors = compute_sun_vectors()   # worst-case western sun

layout = place_buildings_aligned(
    [{'base_boundary': base_boundary('I', 360.0, depth=14.0), 'use': 'commercial'},
     {'base_boundary': base_boundary('L', 620.0, depth=12.0), 'use': 'residential'}],
    SITE, grid, site_setbacks=SET, reference_line=ref_line,
    sun_vectors=sun_vectors, sun_weight=0.5, min_separation=6.0,
)
print('placed:', layout['placed_count'])
for b in layout['buildings']:
    print(f"  building {b['building_index']} ({b['use']:>11}): align={b['alignment_score']:.3f}  "
          f"view={b['unblocked_view_score']:.2f}  combined={b['combined_score']:.3f}")

fig, ax = plt.subplots(figsize=(9, 8))
draw_site(ax); draw_grid(ax, grid, nodes=False)
ax.plot([ref_line[0][0], ref_line[1][0]], [ref_line[0][1], ref_line[1][1]], color='#e67e22', lw=4, zorder=4, label='chosen frontage')
for b, c in zip(layout['buildings'], ('#c0392b', '#2980b9')):
    ax.add_patch(plt.Polygon([(p[0], p[1]) for p in b['boundary'][:-1]], fc=c, ec='k', lw=1.5, alpha=0.75, zorder=6))
    cen = b['centroid_xy']; ax.text(cen[0], cen[1], b['use'], ha='center', fontsize=8, weight='bold', zorder=7)
ax.legend(loc='upper right'); ax.set_title('Two aligned buildings \u2014 commercial on the frontage, residential set back')
ax.set_xlim(-25, 165); ax.set_ylim(-15, 140)
plt.show()